In [ ]:
import numpy as np

class InteractiveEVPlacementGWO:
    def __init__(self):
        print(" EV Charging Station Placement via GWO ")
        print("Please enter the parameters below to initialize your city scenario:\n")

        self.M = int(input("Enter number of Candidate Station Locations (e.g., 10): "))
        self.N = int(input("Enter number of Regional Demand Zones / Neighborhoods (e.g., 25): "))
        self.budget_cap = float(input("Enter maximum allowable Total Budget ($) (e.g., 400000): "))

        print("\n--- Multi-Objective Weights (Must sum up to 1.0) ---")
        self.w1 = float(input("Weight for Installation Cost (0.0 to 1.0): "))
        self.w2 = float(input("Weight for User Travel Distance (0.0 to 1.0): "))
        self.w3 = float(input("Weight for Uncovered Demand Penalty (0.0 to 1.0): "))

        print("\n--- Operational Constraints ---")
        self.max_coverage_radius = float(input("Enter Max Driving Radius (km) users will travel to a charger (e.g., 30): "))

        print("\n--- GWO Algorithm Engine Setup ---")
        self.pop_size = int(input("Enter Wolf Pack Population Size (e.g., 40): "))
        self.max_iter = int(input("Enter Maximum Optimization Iterations (e.g., 50): "))

        np.random.seed(42)
        self.candidate_coords = np.random.rand(self.M, 2) * 100
        self.demand_coords = np.random.rand(self.N, 2) * 100

        self.installation_costs = np.random.randint(20, 90, size=self.M) * 1000
        self.demand_volumes = np.random.randint(5, 30, size=self.N)

        self.dist_matrix = np.linalg.norm(
            self.demand_coords[:, np.newaxis, :] - self.candidate_coords[np.newaxis, :, :],
            axis=2
        )

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def calculate_fitness(self, position):
        binary_decision = (self._sigmoid(position) > 0.5).astype(int)

        if np.sum(binary_decision) == 0:
            return float("inf")

        total_cost = np.sum(binary_decision * self.installation_costs)

        if total_cost > self.budget_cap:
            return float("inf")

        active_stations = np.where(binary_decision == 1)[0]
        active_distances = self.dist_matrix[:, active_stations]

        min_distances = np.min(active_distances, axis=1)

        total_travel_distance = np.sum(min_distances * self.demand_volumes)

        uncovered_mask = min_distances > self.max_coverage_radius
        uncovered_penalty = np.sum(uncovered_mask * self.demand_volumes * 10000)

        fitness = (self.w1 * total_cost) + (self.w2 * total_travel_distance) + (self.w3 * uncovered_penalty)
        return fitness

    def run_optimization(self):

        print(" INITIALIZING GREY WOLF SEARCH MATRICES...")


        X = np.random.uniform(-4, 4, (self.pop_size, self.M))

        alpha_pos, alpha_score = np.zeros(self.M), float("inf")
        beta_pos, beta_score = np.zeros(self.M), float("inf")
        delta_pos, delta_score = np.zeros(self.M), float("inf")

        for t in range(self.max_iter):
            for i in range(self.pop_size):
                fitness = self.calculate_fitness(X[i])

                if fitness < alpha_score:
                    alpha_score = fitness
                    alpha_pos = X[i].copy()
                elif fitness < beta_score and fitness > alpha_score:
                    beta_score = fitness
                    beta_pos = X[i].copy()
                elif fitness < delta_score and fitness > beta_score:
                    delta_score = fitness
                    delta_pos = X[i].copy()

            a = 2 - t * (2 / self.max_iter)

            for i in range(self.pop_size):
                for j in range(self.M):
                    A1, C1 = 2 * a * np.random.rand() - a, 2 * np.random.rand()
                    D_alpha = abs(C1 * alpha_pos[j] - X[i, j])
                    X1 = alpha_pos[j] - A1 * D_alpha

                    A2, C2 = 2 * a * np.random.rand() - a, 2 * np.random.rand()
                    D_beta = abs(C2 * beta_pos[j] - X[i, j])
                    X2 = beta_pos[j] - A2 * D_beta

                    A3, C3 = 2 * a * np.random.rand() - a, 2 * np.random.rand()
                    D_delta = abs(C3 * delta_pos[j] - X[i, j])
                    X3 = delta_pos[j] - A3 * D_delta

                    X[i, j] = (X1 + X2 + X3) / 3

            if (t + 1) % 10 == 0 or t == 0:
                print(f"Iteration {t+1}/{self.max_iter} -> Best Combined Multi-Objective Cost: {alpha_score:.2f}")

        best_layout = (self._sigmoid(alpha_pos) > 0.5).astype(int)
        self._print_results(best_layout, alpha_score)

    def _print_results(self, layout, score):

        print(" OPTIMIZED GRID PLACEMENT SUMMARY")

        if score == float("inf"):
            print(" Failure: GWO could not find a layout that fits within your budget limits.")
            return

        print(f"Binary Deployment Array Result: {layout}")
        print(f"Total Charging Hubs Constructed: {np.sum(layout)} out of {self.M} possible sites")

        final_cost = np.sum(layout * self.installation_costs)
        print(f"Calculated Capital Expenditure: ${final_cost:,} (Remaining Budget: ${self.budget_cap - final_cost:,})")

        print("\nSelected Construction Site Layout Specifics:")
        for idx, build in enumerate(layout):
            if build == 1:
                print(f" -> Site index #{idx} | Base Station Cost: ${self.installation_costs[idx]:,}")

if __name__ == "__main__":
    app = InteractiveEVPlacementGWO()
    app.run_optimization()

 EV Charging Station Placement via GWO 
Please enter the parameters below to initialize your city scenario:

Enter number of Candidate Station Locations (e.g., 10): 10
Enter number of Regional Demand Zones / Neighborhoods (e.g., 25): 25
Enter maximum allowable Total Budget ($) (e.g., 400000): 400000

--- Multi-Objective Weights (Must sum up to 1.0) ---
Weight for Installation Cost (0.0 to 1.0): .4
Weight for User Travel Distance (0.0 to 1.0): .4
Weight for Uncovered Demand Penalty (0.0 to 1.0): .2

--- Operational Constraints ---
Enter Max Driving Radius (km) users will travel to a charger (e.g., 30): 30

--- GWO Algorithm Engine Setup ---
Enter Wolf Pack Population Size (e.g., 40): 40
Enter Maximum Optimization Iterations (e.g., 50): 50
 INITIALIZING GREY WOLF SEARCH MATRICES...
Iteration 1/50 -> Best Combined Multi-Objective Cost: 153021.26
Iteration 10/50 -> Best Combined Multi-Objective Cost: 149264.20
Iteration 20/50 -> Best Combined Multi-Objective Cost: 149264.20
Iteration 30/50